# Taller práctico · Python y Pandas con Palmer Penguins

## 0. Preparación y descarga
1. Si faltara Pandas, instala desde una terminal: `python -m pip install pandas ipykernel` (o ejecuta `%pip install pandas` en una celda).
2. CSV en Kaggle: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

**Comprobación:** debe verse `penguins_size.csv` en el explorador de archivos junto a este notebook.

## 1. Funciones y `lambda`
*`def`* permite reutilizar una operación; *`lambda`* representa una función corta de una expresión. Mostrarlas solo como introducción a transformaciones de columnas.

In [1]:
def a_kilos(gramos):
    return gramos / 1000

conversion = lambda gramos: gramos / 1000

print(a_kilos(3750))
print(conversion(4200))

3.75
4.2


**Salida esperada:** `3.75` y `4.2`.

## 2. Abrir e inspeccionar un dataset
Una **fila** es una observación; una **columna**, una variable. `head`, `shape` e `info` ayudan a reconocer la tabla. El archivo se carga con Pandas, no con Excel.

In [9]:
import pandas as pd

df = pd.read_csv("penguins_size.csv")
print("Dimensiones:", df.shape)
display(df.head(3))
df.info()

Dimensiones: (344, 7)


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE


<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   culmen_length_mm   342 non-null    float64
 3   culmen_depth_mm    342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 24.9 KB


## 3. Seleccionar columnas y posiciones
- `df[[...]]` elige columnas por nombre.
- `iloc[filas, columnas]` elige según la **posición**; el extremo final del slicing no se incluye.
- `unique()` identifica valores diferentes.

In [3]:
display(df[["species", "body_mass_g"]].head(3))
display(df.iloc[0:3, 0:3])
print("Especies:", df["species"].unique())

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0
2,Adelie,3250.0


,species,island,culmen_length_mm
0,Adelie,Torgersen,39.1
1,Adelie,Torgersen,39.5
2,Adelie,Torgersen,40.3


Especies: <ArrowStringArray>
['Adelie', 'Gentoo', 'Chinstrap']
Length: 3, dtype: str


## 4. Filtrar registros con una máscara
Una máscara produce `True` o `False` para cada fila. Para combinar condiciones en Pandas usa `&` (Y) o `|` (O) y **paréntesis** alrededor de cada condición.

In [4]:
mascara = (df["species"] == "Gentoo") & (df["body_mass_g"] > 5000)
seleccion = df.loc[mascara, ["species", "body_mass_g"]]
print("Coincidencias:", len(seleccion))
display(seleccion.head(3))

Coincidencias: 61


,species,body_mass_g
153,Gentoo,5700.0
155,Gentoo,5700.0
156,Gentoo,5400.0


**Salida esperada:** `Coincidencias: 61`. `loc` selecciona filas mediante una condición y las columnas por sus nombres.

## 5. Agrupar y resumir
`groupby` responde una pregunta distinta de `unique`: **calcula resultados para cada categoría**.

In [5]:
print("Registros por especie:")
print(df["species"].value_counts())
print("\nMasa promedio por especie (g):")
print(df.groupby("species")["body_mass_g"].mean().round(1))

Registros por especie:
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

Masa promedio por especie (g):
species
Adelie       3700.7
Chinstrap    3733.1
Gentoo       5076.0
Name: body_mass_g, dtype: float64


**Salida esperada, promedios:** Adelie ≈ `3700.7`, Chinstrap ≈ `3733.1`, Gentoo ≈ `5076.0` gramos. `mean()` omite valores numéricos faltantes.

## 6. Identificar, quitar y rellenar datos faltantes
- `isna().sum()` cuenta celdas vacías por columna.
- `drop()` elimina columnas o filas según lo que indiques.
- `dropna()` descarta filas con faltantes.
- `fillna()` rellena faltantes. **No son operaciones equivalentes:** se elige según el análisis.

Trabaja sobre copias para conservar `df` sin cambios.

In [6]:
print("Faltantes por columna:")
print(df.isna().sum())

sin_masa_vacia = df.dropna(subset=["body_mass_g"])
sin_sexo = df.drop(columns=["sex"])

rellenado = df.copy()
rellenado["body_mass_g"] = rellenado["body_mass_g"].fillna(rellenado["body_mass_g"].median())

print("Filas sin masa vacía:", len(sin_masa_vacia))
print("Columnas después de drop:", sin_sexo.shape[1])
print("Faltantes de masa tras fillna:", rellenado["body_mass_g"].isna().sum())

Faltantes por columna:
species               0
island                0
culmen_length_mm      2
culmen_depth_mm       2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
Filas sin masa vacía: 342
Columnas después de drop: 6
Faltantes de masa tras fillna: 0


**Salida esperada:** `342` filas con masa registrada; `6` columnas tras eliminar `sex`; `0` faltantes en `body_mass_g` después del relleno. **Advertencia:** rellenar con la mediana es solo una demostración, no una decisión universal de limpieza.

## 7. Transformar columnas, `map`, `apply` y exportar
Una operación numérica puede aplicarse directamente a toda una columna. `map` sirve para asignar valores por correspondencia; `apply` permite usar una función sobre cada elemento de una serie. Primero prefiere la operación directa cuando exista.

In [8]:
trabajo = df.copy()
trabajo["masa_kg"] = trabajo["body_mass_g"] / 1000
trabajo["isla_codigo"] = trabajo["island"].map({"Biscoe": "B", "Dream": "D", "Torgersen": "T"})
trabajo["masa_kg_con_funcion"] = trabajo["body_mass_g"].apply(lambda g: g / 1000)

trabajo["categoria_masa"] = "Hasta 5 kg"
trabajo.loc[trabajo["body_mass_g"] > 5000, "categoria_masa"] = "Más de 5 kg"

trabajo.to_csv("pinguinos_procesados.csv", index=False)
display(trabajo[["species", "masa_kg", "isla_codigo", "categoria_masa"]].head(3))
print("Archivo exportado: pinguinos_procesados.csv")

,species,masa_kg,isla_codigo,categoria_masa
0,Adelie,3.75,T,Hasta 5 kg
1,Adelie,3.80,T,Hasta 5 kg
2,Adelie,3.25,T,Hasta 5 kg


Archivo exportado: pinguinos_procesados.csv


**Salida esperada:** las primeras masas son `3.75`, `3.8`, `3.25` kg; se crea `pinguinos_procesados.csv` al lado del notebook.


---
## Anexo
Estas operaciones se incluyen como referencia y desafíos de ampliación. **`concat` agrega tablas por filas o columnas; `merge` relaciona tablas mediante una clave**.

### A. `concat`: juntar subconjuntos verticalmente

In [12]:
parte_a = df.iloc[0:3]
parte_b = df.iloc[3:6]
juntos = pd.concat([parte_a, parte_b], ignore_index=True)
print("Filas combinadas:", juntos.shape[0])

print(parte_a)
print(parte_b)
print(juntos)

Filas combinadas: 6
  species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen              39.1             18.7              181.0   
1  Adelie  Torgersen              39.5             17.4              186.0   
2  Adelie  Torgersen              40.3             18.0              195.0   

   body_mass_g     sex  
0       3750.0    MALE  
1       3800.0  FEMALE  
2       3250.0  FEMALE  
  species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
3  Adelie  Torgersen               NaN              NaN                NaN   
4  Adelie  Torgersen              36.7             19.3              193.0   
5  Adelie  Torgersen              39.3             20.6              190.0   

   body_mass_g     sex  
3          NaN     NaN  
4       3450.0  FEMALE  
5       3650.0    MALE  
  species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen              39.1             18.7              181.0  

**Salida esperada:** `6` filas.

### B. `merge`: relacionar datos mediante una columna común
Los códigos son **ficticios y didácticos**, no variables ecológicas reales.

In [19]:
codigos = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "codigo": ["A", "C", "G"]
})

print(codigos)

combinado = df.merge(codigos, on="species", how="left")
print("Dimensiones tras merge:", combinado.shape)
display(combinado[["species", "codigo"]].head(3))

     species codigo
0     Adelie      A
1  Chinstrap      C
2     Gentoo      G
Dimensiones tras merge: (344, 8)


,species,codigo
0,Adelie,A
1,Adelie,A
2,Adelie,A


**Salida esperada:** `(344, 8)`. Se agrega una columna `codigo` sin duplicar las observaciones porque cada especie aparece solo una vez en la tabla auxiliar.